# 📱 SMS Spam Detection — CSE-4218
**Bangladesh University of Professionals**  
**Team:** Sadia Iffat · Hamza Bin Arif · Faizah Mehnaz · Raiyan Bin Sarwar  

---
## Project Overview
Comparing **Naive Bayes** and **Logistic Regression** for SMS spam detection, then benchmarking against a **free zero-shot LLM** (facebook/bart-large-mnli).

## Cell 1 — Install & Import Libraries

In [ ]:
# Run this ONLY if packages aren't installed yet
# !pip install pandas numpy scikit-learn matplotlib seaborn nltk transformers torch joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import nltk, re, time, joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, classification_report)

nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',  quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print('✅ All imports successful!')

## Cell 2 — Load Dataset
**Download the dataset first:**  
1. Go to: https://archive.ics.uci.edu/dataset/228/sms+spam+collection  
2. Download and extract  
3. Copy `SMSSpamCollection` (no extension) into the `data/` folder

In [ ]:
df = pd.read_csv(
    'data/SMSSpamCollection',
    sep='\t',
    header=None,
    names=['label', 'message'],
    encoding='latin-1'
)

# Convert labels to numbers: ham=0, spam=1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

print(f'Total messages : {len(df)}')
print(f'Ham (legit)    : {(df["label"]=="ham").sum()}')
print(f'Spam           : {(df["label"]=="spam").sum()}')
df.head()

## Cell 3 — Visualise Class Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
counts = df['label'].value_counts()
ax1.bar(counts.index, counts.values,
        color=['#378ADD', '#E24B4A'], width=0.5, edgecolor='white')
ax1.set_title('Class Distribution', fontweight='bold')
ax1.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax1.text(i, v + 30, str(v), ha='center', fontweight='bold')

# Pie chart
ax2.pie(counts.values, labels=counts.index,
        colors=['#378ADD', '#E24B4A'],
        autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax2.set_title('Spam vs Ham Ratio', fontweight='bold')

plt.suptitle('Dataset Overview — UCI SMS Spam Collection',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Cell 4 — Text Preprocessing
Cleans each message through 5 steps:
1. **Lowercase** — `'FREE'` → `'free'`
2. **URL removal** — `'http://win.com'` → `'url'`
3. **Number replacement** — `'1000'` → `'num'`
4. **Stop-word removal** — drops `'the'`, `'is'`, `'at'`
5. **Lemmatisation** — `'winning'` → `'win'`

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', 'url', text)
    text = re.sub(r'\d+', 'num', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w)
              for w in tokens
              if w not in stop_words and len(w) > 1]
    return ' '.join(tokens)

df['clean_msg'] = df['message'].apply(clean_text)

# Show before/after for 3 spam examples
spam_samples = df[df['label']=='spam'][['message','clean_msg']].head(3)
for _, row in spam_samples.iterrows():
    print('BEFORE:', row['message'])
    print('AFTER :', row['clean_msg'])
    print()

## Cell 5 — Train/Test Split & TF-IDF
> **Why TF-IDF?** It converts text into numbers. Words rare in ham but common in spam (like `'free'`, `'win'`, `'prize'`) get high scores.

> **Why split first, then vectorise?** To avoid *data leakage* — the model must never peek at test data during training.

In [ ]:
X = df['clean_msg']
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set : {len(X_train)} messages')
print(f'Test set     : {len(X_test)} messages')

# TF-IDF with unigrams + bigrams (word pairs add context)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)  # fit ONLY on train
X_test_tfidf  = tfidf.transform(X_test)        # transform test

print(f'Feature matrix : {X_train_tfidf.shape[0]} messages × {X_train_tfidf.shape[1]} features')

## Cell 6 — Train Naive Bayes

In [ ]:
t0 = time.time()
nb_model = MultinomialNB(alpha=0.1)   # alpha=smoothing parameter
nb_model.fit(X_train_tfidf, y_train)
nb_time  = time.time() - t0
nb_preds = nb_model.predict(X_test_tfidf)

print(f'Training time : {nb_time*1000:.2f} ms')
print()
print('=== Naive Bayes Results ===')
print(classification_report(y_test, nb_preds, target_names=['Ham','Spam']))

## Cell 7 — Train Logistic Regression

In [ ]:
t0 = time.time()
lr_model = LogisticRegression(max_iter=1000, C=10, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
lr_time  = time.time() - t0
lr_preds = lr_model.predict(X_test_tfidf)

print(f'Training time : {lr_time*1000:.2f} ms')
print()
print('=== Logistic Regression Results ===')
print(classification_report(y_test, lr_preds, target_names=['Ham','Spam']))

## Cell 8 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, preds, name, cmap in zip(
        axes,
        [nb_preds,     lr_preds],
        ['Naive Bayes','Logistic Regression'],
        ['Blues',      'Reds']):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap=cmap,
                linewidths=0.5,
                xticklabels=['Ham (predicted)', 'Spam (predicted)'],
                yticklabels=['Ham (actual)',    'Spam (actual)'],
                annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_title(f'{name} — Confusion Matrix', fontweight='bold', pad=12)

    # Annotate cells
    tn, fp, fn, tp = cm.ravel()
    ax.text(0.5, -0.18,
            f'TN={tn}  FP={fp}  FN={fn}  TP={tp}',
            transform=ax.transAxes, ha='center', fontsize=9,
            color='gray')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 9 — Metrics Comparison Chart

In [ ]:
def get_metrics(y_true, y_pred, name):
    return {
        'Model'    : name,
        'Accuracy' : accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall'   : recall_score(y_true, y_pred),
        'F1-Score' : f1_score(y_true, y_pred),
    }

results = pd.DataFrame([
    get_metrics(y_test, nb_preds, 'Naive Bayes'),
    get_metrics(y_test, lr_preds, 'Logistic Regression'),
])
print(results.round(4).to_string(index=False))

# Plot
metrics = ['Accuracy','Precision','Recall','F1-Score']
nb_vals  = results[results['Model']=='Naive Bayes'][metrics].values[0]
lr_vals  = results[results['Model']=='Logistic Regression'][metrics].values[0]

x = np.arange(len(metrics))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
b1 = ax.bar(x - w/2, nb_vals, w, label='Naive Bayes',
            color='#378ADD', edgecolor='white')
b2 = ax.bar(x + w/2, lr_vals, w, label='Logistic Regression',
            color='#E24B4A', edgecolor='white')
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
ax.set_title('Model Performance Comparison', fontweight='bold', fontsize=13)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0.88, 1.02); ax.set_ylabel('Score')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 10 — Top Spam Keywords
Shows which words Naive Bayes learnt are most associated with spam.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

feature_names = tfidf.get_feature_names_out()

# Top spam keywords
spam_log  = nb_model.feature_log_prob_[1]
top_spam  = spam_log.argsort()[-15:][::-1]
ax1.barh([feature_names[i] for i in top_spam][::-1],
         [spam_log[i] for i in top_spam][::-1],
         color='#E24B4A', alpha=0.85)
ax1.set_title('Top 15 SPAM Keywords (Naive Bayes)', fontweight='bold')
ax1.set_xlabel('Log Probability')

# Top ham keywords
ham_log   = nb_model.feature_log_prob_[0]
top_ham   = ham_log.argsort()[-15:][::-1]
ax2.barh([feature_names[i] for i in top_ham][::-1],
         [ham_log[i] for i in top_ham][::-1],
         color='#378ADD', alpha=0.85)
ax2.set_title('Top 15 HAM Keywords (Naive Bayes)', fontweight='bold')
ax2.set_xlabel('Log Probability')

plt.tight_layout()
plt.savefig('top_keywords.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 11 — Free Zero-Shot LLM (HuggingFace)
> **100% Free** — downloads the model locally, no API key needed.  
> Model: `facebook/bart-large-mnli` (~265 MB, downloaded once)  
> **Note:** Runs on CPU. Testing 100 messages takes ~5–10 minutes.

In [ ]:
from transformers import pipeline

print('Loading model (first time downloads ~265MB)...')
zs = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=-1   # CPU; use device=0 if you have a GPU
)

def zs_predict(text):
    res = zs(text,
             candidate_labels=['spam', 'ham'],
             hypothesis_template='This SMS message is {}.')
    return 1 if res['labels'][0] == 'spam' else 0

# Sample 100 messages (full set = ~30 min on CPU)
SAMPLE = 100
X_sample = X_test.tolist()[:SAMPLE]
y_sample = y_test.tolist()[:SAMPLE]

print(f'Classifying {SAMPLE} messages...')
t0 = time.time()
zs_preds = [zs_predict(m) for m in X_sample]
zs_elapsed = time.time() - t0

print(f'\nZero-Shot (BART) — {SAMPLE} messages in {zs_elapsed:.0f}s')
print(classification_report(y_sample, zs_preds, target_names=['Ham','Spam']))

## Cell 12 — Final Results Table (All 3 Models)

In [ ]:
# Combine all results
final = pd.DataFrame([
    get_metrics(y_test,   nb_preds, 'Naive Bayes (full test)'),
    get_metrics(y_test,   lr_preds, 'Logistic Regression (full test)'),
    get_metrics(y_sample, zs_preds, f'Zero-Shot BART ({SAMPLE} msgs)'),
])
print(final.round(4).to_string(index=False))

# Speed comparison
print(f'\nSpeed Comparison:')
print(f'  Naive Bayes       : {nb_time*1000:.1f} ms')
print(f'  Logistic Regr.    : {lr_time*1000:.1f} ms')
print(f'  Zero-Shot BART    : {zs_elapsed/SAMPLE*1000:.0f} ms per message')

## Cell 13 — Test Your Own Messages

In [ ]:
my_messages = [
    'Congratulations! You WON a FREE iPhone. Claim NOW at bit.ly/xyz',
    'Hey, are we still on for dinner at 7?',
    'URGENT: Your bank account suspended. Verify: secure.fakesite.com',
    'Can you send me the lecture notes?',
    'Win 1000 cash prize! Text WIN to 80085 NOW!',
]

clean_my = [clean_text(m) for m in my_messages]
my_tfidf = tfidf.transform(clean_my)
nb_my    = nb_model.predict(my_tfidf)
lr_my    = lr_model.predict(my_tfidf)

label_map = {0: '✅ HAM', 1: '🚫 SPAM'}
print(f'{"Message":<50} {"NB":>10} {"LR":>10}')
print('-'*72)
for msg, nb_p, lr_p in zip(my_messages, nb_my, lr_my):
    short = msg[:48]+'...' if len(msg)>50 else msg
    print(f'{short:<50} {label_map[nb_p]:>10} {label_map[lr_p]:>10}')

## Cell 14 — Save Models

In [ ]:
joblib.dump(nb_model, 'nb_model.pkl')
joblib.dump(lr_model, 'lr_model.pkl')
joblib.dump(tfidf,    'tfidf_vectorizer.pkl')
print('✅ Models saved!')
print('Files created: nb_model.pkl, lr_model.pkl, tfidf_vectorizer.pkl')